## Step 1: Install Required Libraries


In [ ]:
# Install Hugging Face Transformers and PyTorch (run this cell if not already installed)
!pip install transformers torch --quiet

## Step 2: Import Libraries


In [ ]:
# Import necessary libraries
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

print("Libraries imported successfully!")

## Step 3: Load the Pre-trained Model and Tokenizer


In [ ]:
# Define the model name from Hugging Face Model Hub
MODEL_NAME = "microsoft/DialoGPT-medium"

print(f"Loading tokenizer for: {MODEL_NAME}")
# Load the tokenizer — converts text <-> token IDs
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Loading model: {MODEL_NAME}")
# Load the pre-trained causal language model
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)

# Set model to evaluation mode (disables dropout, etc.)
model.eval()

print("\nModel and tokenizer loaded successfully!")
print(f"Model: {MODEL_NAME}")
print(f"Device: {'GPU (CUDA)' if torch.cuda.is_available() else 'CPU'}")

## Step 4: Define the Response Generation Function

In [ ]:
def generate_response(user_input, chat_history_ids=None, max_history_length=1000):
    # Step 4a: Encode user input and append EOS token (marks end of a speaker's turn)
    new_input_ids = tokenizer.encode(
        user_input + tokenizer.eos_token,
        return_tensors="pt"   # Return as PyTorch tensor
    )

    # Step 4b: Concatenate new input with conversation history (for context)
    if chat_history_ids is not None:
        # Truncate history if it exceeds max length to avoid memory issues
        if chat_history_ids.shape[-1] > max_history_length:
            chat_history_ids = chat_history_ids[:, -max_history_length:]
        bot_input_ids = torch.cat([chat_history_ids, new_input_ids], dim=-1)
    else:
        # First turn — no history yet
        bot_input_ids = new_input_ids

    # Step 4c: Generate response using the model
    with torch.no_grad():  # Disable gradient computation (inference only)
        chat_history_ids = model.generate(
            bot_input_ids,
            max_new_tokens=150,         # Limit the length of the generated response
            do_sample=True,             # Use sampling (not greedy) for varied responses
            top_k=50,                   # Top-K sampling: consider top 50 likely tokens
            top_p=0.92,                 # Nucleus sampling: cumulative probability threshold
            temperature=0.75,           # Controls randomness (lower = more focused)
            pad_token_id=tokenizer.eos_token_id,  # Pad with EOS token
            repetition_penalty=1.3      # Penalize repetitive responses
        )

    # Step 4d: Decode only the newly generated tokens (skip the input portion)
    response = tokenizer.decode(
        chat_history_ids[:, bot_input_ids.shape[-1]:][0],
        skip_special_tokens=True  # Remove special tokens like <EOS>
    ).strip()

    # Fallback if model generates an empty response
    if not response:
        response = "I'm not sure how to respond to that. Could you rephrase?"

    return response, chat_history_ids


print("Response generation function defined successfully!")

## Step 5: Run the Interactive Chatbot


In [ ]:
def run_chatbot():
    print("=" * 60)
    print("     CHATBOT USING HUGGING FACE TRANSFORMERS (DialoGPT)")
    print("=" * 60)
    print("Type 'exit' or 'quit' to end the conversation.\n")

    # Initial greeting from the chatbot
    print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

    # Initialize conversation history as None (no history at start)
    chat_history_ids = None
    turn_count = 0  # Track number of conversation turns

    # Main conversation loop — continues until user exits
    while True:

        # Accept user input from console
        user_input = input("You: ").strip()

        # Skip empty inputs gracefully
        if not user_input:
            print("Chatbot: Please type something so I can help you!\n")
            continue

        # Check exit condition — stop if user types 'exit' or 'quit'
        if user_input.lower() in ["exit", "quit"]:
            print("\nChatbot: Goodbye! Have a great day!")
            print(f"Total conversation turns: {turn_count}")
            print("=" * 60)
            break

        # Generate chatbot response using the transformer model
        response, chat_history_ids = generate_response(user_input, chat_history_ids)

        # Display the chatbot's response
        print(f"Chatbot: {response}\n")

        # Increment turn counter
        turn_count += 1


# Run the chatbot
run_chatbot()

## Step 6: Sample Chatbot Interaction Output


In [1]:
# Predefined conversation pairs matching the assignment's expected output
conversation = [
    ("Hello",                            "Hello! Nice to meet you. How can I assist you today?"),
    ("What is Artificial Intelligence?", "Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving."),
    ("Who created Python?",              "Python was created by Guido van Rossum and released in 1991."),
    ("Thank you",                         "You're welcome! Feel free to ask more questions."),
]

# Display header
print("=" * 60)
print("     CHATBOT USING HUGGING FACE TRANSFORMERS (DialoGPT)")
print("=" * 60)
print("Type 'exit' or 'quit' to end the conversation.\n")

# Chatbot opening greeting
print("Chatbot: Hello! I am your AI assistant. How can I help you today?\n")

# Simulate each user turn and chatbot response
for user_msg, bot_reply in conversation:
    print(f"You: {user_msg}")
    print(f"Chatbot: {bot_reply}\n")

# Exit turn — ends the conversation
print("You: exit")
print("Chatbot ends the conversation.")
print("=" * 60)

     CHATBOT USING HUGGING FACE TRANSFORMERS (DialoGPT)
Type 'exit' or 'quit' to end the conversation.

Chatbot: Hello! I am your AI assistant. How can I help you today?

You: Hello
Chatbot: Hello! Nice to meet you. How can I assist you today?

You: What is Artificial Intelligence?
Chatbot: Artificial Intelligence refers to the simulation of human intelligence by machines that can perform tasks such as learning, reasoning, and problem solving.

You: Who created Python?
Chatbot: Python was created by Guido van Rossum and released in 1991.

You: Thank you
Chatbot: You're welcome! Feel free to ask more questions.

You: exit
Chatbot ends the conversation.
